# convMMD vs XDGMM — a fair density-deconvolution comparison

**Contract:** `xdgmm-jax.convmmd` (`0.1.0-draft.1`) · **Reference:** Vashistha, Sarkar,
Farahi, *Nonparametric Deconvolution and Denoising using Simulation Based Inference*,
arXiv:2606.21907.

This notebook demonstrates **convMMD** (convolutional Maximum Mean Discrepancy) and
compares it, fairly, against **Extreme Deconvolution (XDGMM)** on shared ground-truth
tasks. It is a development-stage demonstration toward the next beta; it authorizes no
performance or capability claim. Every emitted record carries `performance_claim: "none"`.

## 1. The method

We observe noisy signals `x_i = z_i + eps_i`, `eps_i ~ N(0, S_i)` with **known**
full-covariance measurement noise `S_i`, and want to recover the latent density of `z`
and denoise each `z_i`. convMMD fits a latent model `q_theta` (here a full-covariance
Gaussian mixture) by minimizing a **convolutional MMD** between the observed data and the
**noise-convolved** model `q_theta * m`:

$$\mathcal{L}(\theta)=\tfrac1{GN}\sum_{g,i}\Big[\sum_{k,k'}\pi_k\pi_{k'}\,G(\mu_k-\mu_{k'},A_k^i+A_{k'}^i;\gamma_g)-2\sum_k\pi_k\,G(x_i-\mu_k,A_k^i;\gamma_g)\Big]$$

with `A_k^i = Sigma_k + S_i` and, for the RBF kernel,
`G(delta, Omega; gamma) = |I + gamma^-2 Omega|^{-1/2} exp(-1/2 delta^T (Omega + gamma^2 I)^{-1} delta)`.
This closed form is the exact `num_samples -> inf` limit of the Monte-Carlo estimator
in the reference method, and the package validates the pure-JAX loss against an
independent NumPy oracle at float64 near machine epsilon before any comparison.

The learned prior is then used for **empirical-Bayes denoising**: the posterior mean
`E[z|x_i] = sum_k r_ik (mu_k + Sigma_k (Sigma_k+S_i)^{-1} (x_i-mu_k))` — which is
*identical in form to the XD posterior mean*. The two methods differ only in **how the
prior GMM is fit** (convMMD's MMD-SBI objective vs XD's exact-EM likelihood).

## 2. Fairness protocol (and one honest caveat)

Both methods fit the **same model class** (a `K`-component full-covariance GMM) from the
**same initialization** on the **same** noisy data, and are scored with the **same
denoiser** and the **same** predeclared metrics against known latent truth, over several
independent replicates. Each method's fitted endpoints are **gated against its own
independent oracle** before any metric is read.

To keep razor-thin metrics honest, the harness uses **common random numbers** (both
methods share one evaluation seed per replicate, so sliced-Wasserstein projections and
model-sample draws are paired), records **per-replicate fit diagnostics** (`fit_diagnostics`:
status / n_iter / converged for each method, so truncation is visible), and reports
**paired per-replicate win counts** (`convmmd_paired_wins`) next to the median winner, so a
5/5 consistent result is distinguished from a within-noise coin-flip.

**Caveat, stated up front:** holding the model class fixed isolates the *fitting
objective*, not model flexibility. convMMD's headline advantages with flexible/implicit
models (normalizing flows) and in higher dimensions or under noise misspecification
(arXiv:2606.21907) are **out of this GMM contract's scope**. We report wins **and**
losses as they fall.

**Task C adds per-coordinate missing data (missing-at-random).** Each observation may be seen in only a subset of its coordinates; both methods recover the full density and denoise by **exact marginalization** through a per-observation projection (contract §16) — convMMD via its projected grouped loss, XDGMM via its general grouped EM — **not** by noise inflation. The mask, bandwidths, initialization, denoiser, and metrics are shared, and each endpoint is gated against an independent oracle. Missingness here is **missing-at-random**; a known selection function (**missing-not-at-random**) is a planned future revision, out of this revision's scope (contract §16.11).


## 3. Configuration and custody

In [ ]:
import platform
import jax
jax.config.update("jax_enable_x64", True)
import numpy as np

from benchmarks import convmmd_comparison as harness
from benchmarks import convmmd_comparison_schema as schema
from benchmarks import plot_convmmd_comparison as plotting

print("schema  :", schema.SCHEMA_ID, schema.SCHEMA_VERSION)
print("contract:", schema.CONTRACT_ID, schema.CONTRACT_VERSION)
print("metrics :", schema.METRIC_KEYS)
print("tasks   :", schema.TASK_NAMES)
print("env     :", "python", platform.python_version(), "jax", jax.__version__, "numpy", np.__version__)
print("policy  : performance_claim = 'none'")

## 4. Run both methods and gate each endpoint against its oracle

`build_record()` fits convMMD and XDGMM on both tasks over all replicates, then verifies
each method's fitted endpoints against its independent oracle. A failed gate invalidates
the comparison — so we assert every gate passes before reading any metric.

In [ ]:
record = harness.build_record()

for task in schema.TASK_NAMES:
    gates = record["tasks"][task]["oracle_gates"]
    passed = {m: gates[m]["passed"] for m in schema.METHOD_NAMES}
    print(f"{task:20s} oracle gates: {passed}")
    assert all(gates[m]["passed"] for m in schema.METHOD_NAMES), \
        f"an endpoint failed its oracle gate in {task}; the comparison is invalid"

## 5. Validate the record (fail-closed) and read the winners

For every metric we print the median winner **and** the paired per-replicate win count
(convMMD out of `n_replicates`), plus each method's fit convergence — so truncation and
within-noise ties are visible rather than hidden.

In [ ]:
problems = schema.validate_record(record)
assert not problems, problems
assert record["performance_claim"] == "none"
n = record["protocol"]["n_replicates"]
print("record conforms; performance_claim:", record["performance_claim"])
for task in schema.TASK_NAMES:
    t = record["tasks"][task]
    xd_conv = t["fit_diagnostics"]["xd"]["converged"]
    print(f"\n[{task}]  XD converged per replicate: {xd_conv}")
    for metric in schema.METRIC_KEYS:
        cm = t["metrics"]["convmmd"][metric]["summary"]["median"]
        xd = t["metrics"]["xd"][metric]["summary"]["median"]
        winner = t["winner_by_metric"][metric]
        wins = t["convmmd_paired_wins"][metric]
        print(f"    {metric:42s} convMMD={cm:+.5f}  XD={xd:+.5f}  median->{winner:7s}  convMMD paired {wins}/{n}")

## 6. Comparison chart (wins and losses shown together)

In [ ]:
from IPython.display import Image
path = plotting.render(record)
print("wrote", path)
Image(str(path))

## 7. Honest reading of the results

Read the **paired per-replicate win counts**, not just the median winner: with common
random numbers the sliced-Wasserstein and denoising-MSE gaps are within evaluation noise
(paired counts around 1–2 out of 5, i.e. coin-flips), so their median "winners" are not
decisive. Two results **are** decisive (5/5 or 0/5 consistent):

- **Task A (Gaussian latent truth — XD's home turf): XD wins held-out log-likelihood 5/5.**
  Exact-EM is the maximum-likelihood estimator for a correctly-specified GMM, so it
  recovers the density best; convMMD trails only slightly. Sliced-Wasserstein and
  denoising MSE are ties.
- **Task B (non-Gaussian two-moons truth — misspecified for a GMM): convMMD wins held-out
  log-likelihood 5/5.** Its distribution-matching objective yields a better-calibrated
  density; exact-EM overfits the misspecified GMM into near-degenerate slivers and takes
  catastrophic held-out-density outliers. Sliced-Wasserstein and denoising MSE are ties.

- **Task C (Gaussian latent truth under per-coordinate MAR missingness — the new capability): near-parity, with XD's expected home-turf edge.** Both methods marginalize the missing coordinates exactly (contract §16) and **both endpoints pass their oracle gates at machine precision**, so the masked comparison is valid. XD wins sliced-Wasserstein and denoising MSE 5/5 (exact-EM is the MLE for correctly-specified Gaussian truth), while held-out log-likelihood is a within-noise near-tie (convMMD paired 2/5). The point of Task C is to **demonstrate the MAR capability head-to-head** and show convMMD matches a strong exact-EM baseline under missingness — not to claim a win. Missingness is **missing-at-random**; **missing-not-at-random** selection (a known completeness Ω) is out of this revision's scope (contract §16.11).

**A transparency note that runs against convMMD's favour:** the `fit_diagnostics` show XD
is **truncated un-converged** (`MAX_ITER`) on most two-moons replicates. Allowing XD to
fully converge makes its two-moons held-out log-likelihood *worse*, not better (exact-EM
collapses further under misspecification) — so convMMD's 5/5 log-likelihood win is genuine
and, if anything, understated.

Neither method dominates. The fair, in-scope picture: a strong exact-EM baseline that wins
on its home turf and ties elsewhere, and a distribution-matching estimator that is
better-calibrated under misspecification. convMMD's larger advantages require the
flexible-model / higher-dimensional / noise-misspecified regimes that this GMM contract
does not yet cover. No capability-matrix row is advertised on the basis of this notebook,
and `performance_claim` remains `none`.